# Loss normalization: the gradient on the logits for each method

4 tokens. Rows 0-2 are one response (3 tokens, A = +1). Row 3 is another response (1 token, A = -1).
Only the divisor changes between methods.

In [1]:
import torch
torch.manual_seed(0)

b, V = 4, 3                                          # 4 tokens, 3 classes
logits = torch.randn(b, V)                           # (b, V)
y      = torch.tensor([2, 0, 1, 2])                  # (b,)    sampled token per row
y_hat  = torch.softmax(logits, -1)                   # (b, V)
y_     = torch.nn.functional.one_hot(y, V).float()   # (b, V)
A      = torch.tensor([1.0, 1.0, 1.0, -1.0])         # (b,)
w      = torch.ones(b)                               # (b,)    w = 1, to isolate the divisor

T_of_row = torch.tensor([3.0, 3.0, 3.0, 1.0])        # (b,)    length of each row's response
G, L_max = 2, 4                                      # 2 responses, max length 4

In [2]:
# token-level (DAPO, CISPO paper): divide by b = 4
grad = (w * A).reshape(-1, 1) * (y_hat - y_) / b
grad

tensor([[ 0.2112,  0.0337, -0.2449],
        [-0.0622,  0.0360,  0.0263],
        [ 0.0871, -0.1155,  0.0283],
        [-0.0691, -0.0569,  0.1260]])

In [3]:
# miles default: divide by the response's length
grad = (w * A).reshape(-1, 1) * (y_hat - y_) / T_of_row.reshape(-1, 1)
grad

tensor([[ 0.2815,  0.0450, -0.3265],
        [-0.0830,  0.0479,  0.0350],
        [ 0.1161, -0.1539,  0.0378],
        [-0.2762, -0.2277,  0.5039]])

In [4]:
# GRPO: divide by G * the response's length
grad = (w * A).reshape(-1, 1) * (y_hat - y_) / (G * T_of_row).reshape(-1, 1)
grad

tensor([[ 0.1408,  0.0225, -0.1633],
        [-0.0415,  0.0240,  0.0175],
        [ 0.0581, -0.0770,  0.0189],
        [-0.1381, -0.1138,  0.2520]])

In [5]:
# Dr. GRPO: divide by G * L_max
grad = (w * A).reshape(-1, 1) * (y_hat - y_) / (G * L_max)
grad

tensor([[ 0.1056,  0.0169, -0.1224],
        [-0.0311,  0.0180,  0.0131],
        [ 0.0436, -0.0577,  0.0142],
        [-0.0345, -0.0285,  0.0630]])

Row 3 (the 1-token response) vs rows 0-2 (the 3-token response):
- token-level and Dr. GRPO: every row is divided by the same number.
- miles and GRPO: row 3 is divided by 3x less, so its gradient is 3x larger relative to rows 0-2.